# Confidence gating (L9_partial = L9_cosface + 部分解冻 f=1/3)

**Cell 1-7**：训练流水线(原版),从头训 ~1.5h,导出每样本 test 的 p1(`gating_per_sample.csv`)。
**Cell 8-11(分析)**：人工固定阈值 `TAU=0.999`,逐折+整体表现成表;把进入 Stage 2 却判错的样本导出 CSV 发给 Stage 2 同学。


In [ ]:
# Cell 1. Config — 门控分析(基于已选定模型 L9_partial: L9_cosface + 部分解冻 f=1/3)
import warnings, os, logging, gc, time, re
from contextlib import nullcontext
warnings.filterwarnings("ignore"); logging.getLogger("transformers").setLevel(logging.ERROR)
os.environ["HF_HOME"]="/root/autodl-tmp/hf_cache"; os.environ["HF_ENDPOINT"]="https://hf-mirror.com"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"]="1"; os.environ["HF_HUB_DISABLE_TELEMETRY"]="1"
from pathlib import Path
Path("/root/autodl-tmp/hf_cache").mkdir(parents=True, exist_ok=True)
LOCAL_ONLY=False
SPLIT_ROOT=Path("splits_clip/cv5"); CLASSES_CSV=Path("splits_clip/classes.csv")
IMG_ROOT=Path("."); ABL_FOLD=0
OUTDIR=Path("clip_gating"); OUTDIR.mkdir(exist_ok=True)
VIS_ID, VNAME="microsoft/swinv2-base-patch4-window8-256","swinv2_base"
TXT_ID, TNAME="sentence-transformers/all-mpnet-base-v2","mpnet"
# === 锁定赢家配置 L9_partial ===
FIXED_LOSS="L9_cosface_m04"; SIM="cosine"
ADAPT="partial"; FREEZE_VIS, FREEZE_TXT = 1/3, 1/3
LORA_R, LORA_ALPHA, LORA_DROPOUT = 32, 64, 0.0   # 占位(不用)
PROJ_DIM=512
EPOCHS, MIN_EPOCHS, PATIENCE = 100, 40, 30
BATCH, OOM_BATCH = 16, 4
LR_HEAD, LR_ENC, MAX_TOK, GRAD_CLIP, SEED = 1e-3, 1e-5, 32, 1.0, 42
AUGMENT=True; USE_SAMPLER=True
FOLDS=[0,1,2,3,4]
print(f"OUTDIR={OUTDIR} | 门控模型 = L9_partial (L9_cosface + 部分解冻 f=1/3)")

In [ ]:
# Cell 2. Imports + data + dataset/loader (same as Method 1)
import numpy as np, pandas as pd, random, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image
from torchvision import transforms
from transformers import AutoModel, AutoImageProcessor, AutoTokenizer
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from collections import deque
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
set_seed(SEED)
DEV="cuda" if torch.cuda.is_available() else "cpu"; BF16=(DEV=="cuda" and torch.cuda.is_bf16_supported())
print("device:",DEV,"| bf16:",BF16)

CLASSES=list(pd.read_csv(CLASSES_CSV)["caption"]); CIDX={c:i for i,c in enumerate(CLASSES)}; NCLS=len(CLASSES)
def load_fold(k):
    out={}
    for nm in ["train","val","test"]:
        df=pd.read_csv(SPLIT_ROOT/f"fold{k}"/f"{nm}.csv")
        df["label"]=df["caption_norm"].map(CIDX); out[nm]=df.reset_index(drop=True)
    return out
def metrics(true,pred):
    true=list(true); pred=list(pred)
    labs=sorted(set(true))   # 只在测试集中实际出现的类上算 macro(标准做法,不被缺席类拉低)
    acc=accuracy_score(true,pred)
    pma,rma,fma,_=precision_recall_fscore_support(true,pred,labels=labs,average="macro",zero_division=0)
    pw,rw,fw,_=precision_recall_fscore_support(true,pred,labels=labs,average="weighted",zero_division=0)
    return {"acc":acc,"macroF1":fma,"macroP":pma,"macroR":rma,"weightedF1":fw}

AUG=transforms.Compose([transforms.RandomHorizontalFlip(),
                        transforms.ColorJitter(0.2,0.2,0.2,0.02),
                        transforms.RandomRotation(8)])
class ImgDS(Dataset):
    def __init__(self,df,proc,train):
        self.df=df; self.proc=proc; self.train=train and AUGMENT
        self.y=torch.tensor(df["label"].values)
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        img=Image.open(IMG_ROOT/self.df.iloc[i]["image"]).convert("RGB")
        if self.train: img=AUG(img)
        pix=self.proc(images=img,return_tensors="pt")["pixel_values"][0]
        return pix, self.y[i], i
def vis_pool(enc,pix):
    m=getattr(enc,"vision_model",enc); h=m(pixel_values=pix).last_hidden_state
    if h.dim()==4: h=h.flatten(2).transpose(1,2)
    return h.mean(1)
def make_loader(df,proc,train):
    ds=ImgDS(df,proc,train)
    if train and USE_SAMPLER:
        cnt=df["label"].value_counts(); w=df["label"].map(lambda l:1.0/cnt[l]).values
        sm=WeightedRandomSampler(torch.as_tensor(w,dtype=torch.double),num_samples=len(df),replacement=True)
        return DataLoader(ds,batch_size=BATCH,sampler=sm,num_workers=0)
    return DataLoader(ds,batch_size=BATCH,shuffle=train,num_workers=0)
print("data ready, NCLS=",NCLS)


In [ ]:
# Cell 3. Loss implementations -- the heart of this notebook
class LossFunctions:
    # All losses operate on: logits [B, NCLS] from cosine*scale, labels [B], 
    # image_emb [B, D] normalized, text_emb [NCLS, D] normalized, batch_pos [B, D] = text_emb[labels]
    # class_counts [NCLS] long-tail prior info

    @staticmethod
    def l0_infonce(logits, labels, ie, te, class_counts):
        # symmetric InfoNCE on (image -> text) and (text -> image)
        tpos = te[labels]
        scale = logits.max().item() / max((ie @ te.t())[0,0].item(), 1e-8) if False else None
        # Reconstruct from scratch for clarity
        return F.cross_entropy(logits, labels)  # InfoNCE here = CE over class-prompts when using same logits matrix
        # Note: the real InfoNCE you trained with is symmetric (img->txt + txt->img),
        # but for fold0 ablation we use single-direction CE on the same cosine logits matrix.

    @staticmethod  
    def l0_infonce_symmetric(logits, labels, ie, te, class_counts):
        # True symmetric InfoNCE: image to its positive text + positive text to image
        tpos = te[labels]   # [B, D]
        lc_i2t = logits[:, labels]  # but this is wrong shape; use direct dot
        # Compute pairwise sim between batch_img and batch_pos
        s = ie.size(0)
        lc = (logits[range(s)].t()[labels].t()) if False else None
        # Simplest: standard image-text contrastive with batch-as-negatives
        # logits_i2t[i,j] = sim(img_i, txt_pos_j) where txt_pos_j = text_emb[labels[j]]
        scale = (logits.max() / (ie@te.t()).max()).item() if (ie@te.t()).max()>0 else 100.0
        lc = scale * (ie @ tpos.t())
        tgt = torch.arange(ie.size(0), device=ie.device)
        return 0.5 * (F.cross_entropy(lc, tgt) + F.cross_entropy(lc.t(), tgt))

    @staticmethod
    def l1_ce(logits, labels, ie, te, class_counts):
        return F.cross_entropy(logits, labels)

    @staticmethod
    def l2_balanced_softmax(logits, labels, ie, te, class_counts):
        adj = torch.log(class_counts.float() + 1e-8).to(logits.device).unsqueeze(0)
        return F.cross_entropy(logits + adj, labels)

    @staticmethod
    def l3_logit_adjust(logits, labels, ie, te, class_counts, tau=1.0):
        prior = (class_counts.float() / class_counts.sum()).to(logits.device)
        adj = (tau * torch.log(prior + 1e-8)).unsqueeze(0)
        return F.cross_entropy(logits + adj, labels)

    @staticmethod
    def l4_focal(logits, labels, ie, te, class_counts, gamma=2.0):
        ce = F.cross_entropy(logits, labels, reduction="none")
        p = (-ce).exp()  # p_y = exp(-CE)
        return ((1-p)**gamma * ce).mean()

    @staticmethod
    def l5_cb_loss(logits, labels, ie, te, class_counts, beta=0.99):
        eff_num = (1.0 - beta**class_counts.float()) / (1.0 - beta + 1e-8)
        weights = (1.0 / (eff_num + 1e-8))
        weights = weights / weights.sum() * NCLS  # normalize so mean weight ~ 1
        return F.cross_entropy(logits, labels, weight=weights.to(logits.device))

    @staticmethod
    def l6_ldam(logits, labels, ie, te, class_counts, max_m=0.5, s=30.0):
        m_list = 1.0 / torch.sqrt(torch.sqrt(class_counts.float() + 1e-8))
        m_list = m_list * (max_m / m_list.max())
        m_list = m_list.to(logits.device)
        index = torch.zeros_like(logits, dtype=torch.bool)
        index.scatter_(1, labels.unsqueeze(1), True)
        margin_batch = m_list[labels].unsqueeze(1)
        logits_m = logits - index.float() * margin_batch
        return F.cross_entropy(logits_m * s / logits.max().clamp(min=1.0), labels)

    @staticmethod
    def l7_supcon(logits, labels, ie, te, class_counts, temp=0.07):
        # Supervised contrastive on image features: same-class samples in batch are positives
        z = F.normalize(ie, dim=-1)
        sim = z @ z.t() / temp  # [B, B]
        # mask diagonal
        mask_diag = torch.eye(z.size(0), device=z.device, dtype=torch.bool)
        sim.masked_fill_(mask_diag, -1e9)
        # positive mask: same label, not self
        labels_eq = labels.unsqueeze(0) == labels.unsqueeze(1)
        pos_mask = labels_eq & ~mask_diag
        # log_prob
        logits_max, _ = sim.max(dim=1, keepdim=True)
        sim_norm = sim - logits_max.detach()
        exp_sim = sim_norm.exp()
        log_prob = sim_norm - torch.log(exp_sim.sum(1, keepdim=True) + 1e-12)
        # mean log_prob over positives (skip samples with no positives)
        n_pos = pos_mask.sum(1)
        valid = n_pos > 0
        if valid.sum() == 0: 
            return F.cross_entropy(logits, labels)  # fallback
        mean_log_prob = (pos_mask.float() * log_prob).sum(1)[valid] / n_pos[valid].float()
        loss_supcon = -mean_log_prob.mean()
        # Add the cosine-CE for classifier supervision (otherwise model has no class anchor)
        return 0.5 * loss_supcon + 0.5 * F.cross_entropy(logits, labels)

    @staticmethod
    def l8_arcface(logits, labels, ie, te, class_counts, m=0.5, s=30.0):
        # logits are scale * cosine; recover cosine
        cos_theta = logits / max(logits.abs().max().item(), 1.0)
        cos_theta = cos_theta.clamp(-1+1e-7, 1-1e-7)
        theta = cos_theta.acos()
        # Only apply margin to ground-truth class
        index = torch.zeros_like(logits, dtype=torch.bool)
        index.scatter_(1, labels.unsqueeze(1), True)
        theta_m = theta + index.float() * m
        cos_theta_m = theta_m.cos()
        out = s * cos_theta_m
        return F.cross_entropy(out, labels)

    @staticmethod
    def l9_cosface(logits, labels, ie, te, class_counts, m=0.4, s=30.0):
        cos_theta = logits / max(logits.abs().max().item(), 1.0)
        cos_theta = cos_theta.clamp(-1+1e-7, 1-1e-7)
        index = torch.zeros_like(logits, dtype=torch.bool)
        index.scatter_(1, labels.unsqueeze(1), True)
        cos_theta_m = cos_theta - index.float() * m
        out = s * cos_theta_m
        return F.cross_entropy(out, labels)

LOSS_FN = {
    "L0_infonce":            LossFunctions.l0_infonce_symmetric,
    "L1_ce":                 LossFunctions.l1_ce,
    "L2_balanced_softmax":   LossFunctions.l2_balanced_softmax,
    "L3_logit_adjust_t10":   LossFunctions.l3_logit_adjust,
    "L4_focal_g2":           LossFunctions.l4_focal,
    "L5_cb_loss_b099":       LossFunctions.l5_cb_loss,
    "L6_ldam":               LossFunctions.l6_ldam,
    "L7_supcon":             LossFunctions.l7_supcon,
    "L8_arcface_m05":        LossFunctions.l8_arcface,
    "L9_cosface_m04":        LossFunctions.l9_cosface,
}
print("loss functions:", list(LOSS_FN.keys()))


In [ ]:
# Cell 4. 统一模型:适配 partial(部分解冻,冻前段) / lora;forward 写死 cosine
def partial_unfreeze(model, freeze_frac):
    for p in model.parameters(): p.requires_grad=False
    if freeze_frac>=0.999: return 0
    names=[n for n,_ in model.named_parameters()]
    li=[int(m.group(1)) for n in names for m in [re.search(r"encoder\.layer\.(\d+)\.",n)] if m]
    si=[int(m.group(1)) for n in names for m in [re.search(r"encoder\.layers\.(\d+)\.",n)] if m]
    ntr=0
    if li:
        cut=int(round(max(li)*freeze_frac))
        for n,p in model.named_parameters():
            m=re.search(r"encoder\.layer\.(\d+)\.",n)
            if (m and int(m.group(1))>=cut) or any(k in n.lower() for k in ["layernorm","pooler"]): p.requires_grad=True; ntr+=p.numel()
    elif si:
        cut=int(round((max(si)+1)*freeze_frac))
        for n,p in model.named_parameters():
            m=re.search(r"encoder\.layers\.(\d+)\.",n)
            if (m and int(m.group(1))>=cut) or "layernorm" in n.lower(): p.requires_grad=True; ntr+=p.numel()
    else:
        for p in model.parameters(): p.requires_grad=True; ntr+=p.numel()
    return ntr

class CLIPRetriever(nn.Module):
    def __init__(self, vis_id, txt_id, proc, D=PROJ_DIM):
        super().__init__()
        self.venc=AutoModel.from_pretrained(vis_id,torch_dtype=torch.float32,local_files_only=LOCAL_ONLY)
        self.tenc=AutoModel.from_pretrained(txt_id,torch_dtype=torch.float32,local_files_only=LOCAL_ONLY)
        with torch.no_grad():
            dummy=proc(images=Image.new("RGB",(256,256)),return_tensors="pt")["pixel_values"]
            dv=vis_pool(self.venc,dummy).shape[-1]
        dt=self.tenc.config.hidden_size
        self.ip=nn.Sequential(nn.Linear(dv,D),nn.GELU(),nn.Linear(D,D))
        self.tp=nn.Sequential(nn.Linear(dt,D),nn.GELU(),nn.Linear(D,D))
        self.scale=nn.Parameter(torch.tensor(float(np.log(1/0.07))))
        self._apply_adapt()
    def _apply_adapt(self):
        if ADAPT=="frozen":
            for p in self.venc.parameters(): p.requires_grad=False
            for p in self.tenc.parameters(): p.requires_grad=False
            print("   ADAPT=frozen: 全冻结,只训练投影头")
        elif ADAPT=="partial":
            nv=partial_unfreeze(self.venc,FREEZE_VIS); nt=partial_unfreeze(self.tenc,FREEZE_TXT)
            print(f"   ADAPT=partial vis_f={FREEZE_VIS:.2f}/txt_f={FREEZE_TXT:.2f}: unfrozen vision {nv:,} | text {nt:,}")
        elif ADAPT=="lora":
            from peft import LoraConfig, get_peft_model
            cfg=LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, target_modules="all-linear", bias="none")
            self.venc=get_peft_model(self.venc, cfg); self.tenc=get_peft_model(self.tenc, cfg)
            nv=sum(p.numel() for p in self.venc.parameters() if p.requires_grad)
            nt=sum(p.numel() for p in self.tenc.parameters() if p.requires_grad)
            print(f"   ADAPT=lora(r={LORA_R},a={LORA_ALPHA}): trainable vision {nv:,} | text {nt:,}")
        else: raise ValueError(f"unknown ADAPT={ADAPT}")
    def enc_text(self, ids, mask):
        o=self.tenc(input_ids=ids,attention_mask=mask).last_hidden_state
        m=mask.unsqueeze(-1).float(); return self.tp((o*m).sum(1)/m.sum(1).clamp(min=1))
    def enc_img(self, pix): return self.ip(vis_pool(self.venc,pix))
    def forward_features(self, pix, tids, tmask):
        ie=self.enc_img(pix); te=self.enc_text(tids,tmask)
        s=self.scale.exp().clamp(max=100.0)
        ien=F.normalize(ie,dim=-1); ten=F.normalize(te,dim=-1)
        return s*ien@ten.t(), ien, ten
    @torch.no_grad()
    def predict(self, pix, tids, tmask):
        lg,_,_=self.forward_features(pix,tids,tmask); return lg
print("unified model ready (partial / lora)")

In [ ]:
# Cell 5. Single-loss training function(训完每折保存 checkpoint)
SAVE_CKPT=True
CKPT_DIR=OUTDIR/"ckpts"
def train_run(loss_name, vis_id, txt_id, proc, tok, txt_ids, txt_mask, fold, batch):
    set_seed(SEED)   # 每个 run 从同一随机起点:在建模型/读数据之前重置 -> init+数据顺序对所有配置一致
    F_=load_fold(fold)
    model=CLIPRetriever(vis_id, txt_id, proc).to(DEV)
    head=[p for n,p in model.named_parameters() if p.requires_grad and ("ip." in n or "tp." in n or n=="scale")]
    enc=[p for n,p in model.named_parameters() if p.requires_grad and not("ip." in n or "tp." in n or n=="scale")]
    grps=[{"params":head,"lr":LR_HEAD}]+([{"params":enc,"lr":LR_ENC}] if enc else [])
    opt=torch.optim.AdamW(grps,weight_decay=0.01)

    cnt=F_["train"]["label"].value_counts()
    class_counts=torch.tensor([cnt.get(i,1) for i in range(NCLS)], dtype=torch.long, device=DEV)
    print(f"   class_counts: min={class_counts.min().item()} max={class_counts.max().item()}")

    loss_fn = LOSS_FN[loss_name]
    dl=make_loader(F_["train"],proc,True)
    TI,TM=txt_ids.to(DEV),txt_mask.to(DEV)
    def ev(split):
        model.eval(); P=[]; T=[]
        for pix,y,_ in DataLoader(ImgDS(F_[split],proc,False),batch_size=batch):
            lg=model.predict(pix.to(DEV),TI,TM); P+=lg.argmax(1).cpu().tolist(); T+=y.tolist()
        return T,P

    win=deque(maxlen=3); best=-1; bstate=None; bad=0; bep=-1
    nan_streak=0
    for ep in range(EPOCHS):
        model.train(); run=0; st=0
        for pix,y,_ in dl:
            opt.zero_grad()
            ctx=torch.autocast("cuda",dtype=torch.bfloat16) if BF16 else nullcontext()
            with ctx:
                logits, ie, te = model.forward_features(pix.to(DEV),TI,TM)
                loss = loss_fn(logits, y.to(DEV), ie, te, class_counts)
            if not torch.isfinite(loss): 
                nan_streak += 1
                if nan_streak > 50: 
                    print(f"   ABORT: loss non-finite for 50 batches")
                    return None
                continue
            nan_streak = 0
            loss.backward()
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],GRAD_CLIP)
            opt.step()
            with torch.no_grad(): model.scale.clamp_(max=float(np.log(100.0)))
            run+=loss.item(); st+=1
        T,P=ev("val"); m=metrics(T,P); sc=m["acc"]+m["macroF1"]; win.append(sc); sm=sum(win)/len(win)
        if sm>best: best=sm; bep=ep; bad=0; bstate={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        elif ep>=MIN_EPOCHS: bad+=1
        if ep%5==0 or ep==EPOCHS-1: print(f"   ep{ep:03d} loss={run/max(1,st):.3f} val_acc={m['acc']:.3f} val_mF1={m['macroF1']:.3f}")
        if ep>=MIN_EPOCHS and bad>=PATIENCE: print(f"   early stop @ep{ep} (best @ep{bep})"); break

    if bstate: model.load_state_dict(bstate)
    if SAVE_CKPT:
        CKPT_DIR.mkdir(parents=True, exist_ok=True)
        ckpt_path=CKPT_DIR/f"{loss_name}_{ADAPT}_fold{fold}.pt"
        sd=bstate if bstate else {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        torch.save({"state_dict":sd,"loss":loss_name,"adapt":ADAPT,"fold":fold,
                    "freeze_vis":globals().get("FREEZE_VIS"),"freeze_txt":globals().get("FREEZE_TXT"),
                    "lora":({"r":LORA_R,"alpha":LORA_ALPHA,"dropout":LORA_DROPOUT} if ADAPT=="lora" else None),
                    "vis_id":vis_id,"txt_id":txt_id,"classes":CLASSES,"proj_dim":PROJ_DIM}, ckpt_path)
        print(f"   saved ckpt -> {ckpt_path} ({ckpt_path.stat().st_size/1e6:.0f} MB)")
    model.eval(); P=[]; T=[]; PROBS=[]
    for pix,y,_ in DataLoader(ImgDS(F_["test"],proc,False),batch_size=batch):
        lg=model.predict(pix.to(DEV),TI,TM); pr=torch.softmax(lg.float(),dim=1)
        P+=pr.argmax(1).cpu().tolist(); T+=y.tolist(); PROBS.append(pr.cpu())
    import numpy as _np; PROBS=torch.cat(PROBS,0).numpy()
    fm=metrics(T,P)
    del model,opt,dl; gc.collect(); torch.cuda.empty_cache() if DEV=="cuda" else None
    return fm, T, P, PROBS


In [ ]:
# Cell 6. Helpers + repair-family 解析(支持 'defect__location' 与自然语言两种)
from transformers import AutoTokenizer, AutoImageProcessor
import pandas as pd
def text_tensors(txt_id):
    tok=AutoTokenizer.from_pretrained(txt_id, local_files_only=LOCAL_ONLY)
    e=tok(CLASSES,padding="max_length",truncation=True,max_length=MAX_TOK,return_tensors="pt")
    return tok, e["input_ids"], e["attention_mask"]
def run_combo(vis_id, txt_id, fold):
    proc=AutoImageProcessor.from_pretrained(vis_id, local_files_only=LOCAL_ONLY)
    tok,TI,TM=text_tensors(txt_id)
    try:
        return train_run(FIXED_LOSS, vis_id, txt_id, proc, tok, TI, TM, fold, BATCH)
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            torch.cuda.empty_cache(); gc.collect(); print(f"   OOM -> retry batch {OOM_BATCH}")
            return train_run(FIXED_LOSS, vis_id, txt_id, proc, tok, TI, TM, fold, OOM_BATCH)
        raise

# repair-family:类名含 '__' 取前缀(section_loss__main_girder -> section_loss);否则用关键词
DEFECT_RULES=[
  ("missing_fastener", ["missing rivet","missing bolt","rivet","bolt"]),
  ("crack",            ["crack","fracture"]),
  ("hole_or_pitting",  ["hole","perforation","pitting"]),
  ("jagged_edge",      ["jagged","edge"]),
  ("buckling_deform",  ["buckl","deform","distort","bent","bow"]),
  ("masonry_collapse", ["masonry","brick","collapse"]),
  ("section_loss",     ["corrosion","corroded","section loss","loss"]),
]
def family_of_caption(cap):
    c=str(cap).strip()
    if "__" in c: return c.split("__")[0].strip().lower()
    cl=c.lower()
    for name,kws in DEFECT_RULES:
        if any(k in cl for k in kws): return name
    return "other"
FAMILY_OF_IDX=[family_of_caption(CLASSES[i]) for i in range(NCLS)]
_m=pd.DataFrame({"class":CLASSES,"family":FAMILY_OF_IDX})
print("repair-family 分布:"); print(_m["family"].value_counts().to_string())
_un=_m[_m["family"]=="other"]
if len(_un): print(f"\n[请校对] 落入 other 的类别 {len(_un)} 个(可调 DEFECT_RULES):"); print(_un["class"].to_string(index=False))
print("helpers + family map ready")

In [ ]:
# Cell 7. 训练 L9_partial 5 折,导出每样本 top5 + 置信度 + fam_agree(断点续跑,~1.5h)
import time, pandas as pd, numpy as np
PS=OUTDIR/"gating_per_sample.csv"
COLS=["fold","image","true","pred","top5_idx","p1","p2","fam_agree","correct","fam_true","fam_pred"]
done=set()
if PS.exists(): _d=pd.read_csv(PS); done=set(_d["fold"].unique().tolist())
for fold in FOLDS:
    if fold in done: print(f"[SKIP] fold{fold} 已有"); continue
    print("="*64); print(f"L9_partial | fold {fold}")
    fm,T,P,PROBS=run_combo(VIS_ID,TXT_ID,fold)
    Ftest=load_fold(fold)["test"]
    rows=[]
    for r in range(len(T)):
        pr=PROBS[r]; order=np.argsort(pr)[::-1]; t5=[int(x) for x in order[:5]]
        p1=float(pr[t5[0]]); p2=float(pr[t5[1]]); ft1=FAMILY_OF_IDX[t5[0]]
        fa=float(np.mean([FAMILY_OF_IDX[i]==ft1 for i in t5]))   # top5 同 family 比例
        rows.append({"fold":fold,"image":str(Ftest.iloc[r]["image"]),"true":int(T[r]),"pred":t5[0],
                     "top5_idx":" ".join(map(str,t5)),"p1":round(p1,4),"p2":round(p2,4),
                     "fam_agree":round(fa,4),"correct":int(t5[0]==T[r]),
                     "fam_true":FAMILY_OF_IDX[int(T[r])],"fam_pred":ft1})
    pd.DataFrame(rows)[COLS].to_csv(PS,mode="a",header=not PS.exists(),index=False)
    print(f"   fold{fold} acc={fm['acc']:.4f} | 导出 {len(rows)} 条")
print(">>> 导出完成 ->", PS)

## 门控分析（人工固定阈值，合并全部折）

所有折用同一个人工指定的高置信阈值 `TAU`：p1 ≥ TAU 进自动档(→ Stage 2)，其余转人工。统计逐折+整体，并把进入 Stage 2 但判错的样本导出给 Stage 2 同学。

In [ ]:
# Cell 8. 配置：路径 + 类名 + 维修映射 + 固定阈值
from pathlib import Path
import pandas as pd, numpy as np
OUTDIR = Path("clip_gating")
TAU = 0.999                            # 人工指定的固定高置信阈值（工作点）
CLASS_NAMES = ["corroded drainage spigot at soffit", "disconnected weld on bottom boom upper rolled steel angle on truss outside elevation", "hole and crack in deck at soffit", "hole and pitting in deck plate", "hole in caisson", "hole in deck at soffit", "hole in deck at soffit with existing plate falling off", "hole in deck plate", "hole in rail bearer web", "hole in truss flange", "hole in truss web", "hole in web of diaphragm plate", "jagged edges", "jagged flange on cross girder at soffit", "large hole throughout caisson", "lower rolled steel angle of bottom boom corroded on truss outside elevation", "missing rivet or bolt", "redundant element", "section loss", "section loss in bottom flange at rail bearer", "section loss in bowstring arch n-type truss flange angles", "section loss in crank angle of deck stiffener at soffit", "section loss in crank on main girder external face", "section loss in crank on main girder internal face", "section loss in flange on cross girder", "section loss in flange on main girder external face", "section loss in lower crank angle", "section loss in stiffener angle at rail bearer", "section loss in top boom angle on truss outside elevation", "upper rolled steel angle of bottom boom corroded on truss outside elevation"]
M2REP = {"corroded drainage spigot at soffit": "install new deck plate with spigot", "disconnected weld on bottom boom upper rolled steel angle on truss outside elevation": "re-connect by welding", "hole and crack in deck at soffit": "drill stop hole and cover with welded plate applying a 6mm cfw", "hole and pitting in deck plate": "install a welded 8mm plate over the defect using a 6mm cfw, new plate to extend a minimum of 100mm past the defect area, provision of a spigot to direct the flow of water below structural elements", "hole in caisson": "fill all holes with devcon", "hole in deck at soffit": "cover the damaged area with a metal plate and weld it securely using a 6mm continuous fillet weld", "hole in deck at soffit with existing plate falling off": "cover the damaged area with a metal plate and weld it securely using a 6mm continuous fillet weld", "hole in deck plate": "install a welded 8mm plate over the defect using a 6mm cfw, new plate to extend a minimum of 100mm past the defect area, provision of a spigot to direct the flow of water below structural elements", "hole in rail bearer web": "cover the damaged area with a small metal plate and weld it securely using a 6mm continuous fillet weld", "hole in truss flange": "carry out a web strengthening plate repair similar to form003 design on bottom of truss", "hole in truss web": "cut out corrosion", "hole in web of diaphragm plate": "replace diaphragm plate like for like and connecting angles if necessary", "jagged edges": "knife edge element to minimum of 3mm thickness and apply a 4mm radius", "jagged flange on cross girder at soffit": "grind back corroded section and apply radius", "large hole throughout caisson": "fill all holes with devcon", "lower rolled steel angle of bottom boom corroded on truss outside elevation": "remove a damaged part of the reinforced steel structure and replace it with a new section. connect the new section to the existing steel structure at two points using strong splices", "missing rivet or bolt": "insert new bolt", "redundant element": "remove redundant elements", "section loss": "knife edge angle and apply radius for paint system", "section loss in bottom flange at rail bearer": "knife edge angle and apply radius for paint system", "section loss in bowstring arch n-type truss flange angles": "cut out knife edged steel", "section loss in crank angle of deck stiffener at soffit": "cut out corroded section and weld new piece in position like for like", "section loss in crank on main girder external face": "cut out corroded section", "section loss in crank on main girder internal face": "cut out corroded section", "section loss in flange on cross girder": "cut out corroded section", "section loss in flange on main girder external face": "cut out corroded section", "section loss in lower crank angle": "cut out corroded section and apply radius for paint system", "section loss in stiffener angle at rail bearer": "replace angle like for like", "section loss in top boom angle on truss outside elevation": "replace top boom angle like for like with existing and splice at joints", "upper rolled steel angle of bottom boom corroded on truss outside elevation": "remove a damaged part of the reinforced steel structure and replace it with a new section. connect the new section to the existing steel structure at two points using strong splices"}
test = pd.read_csv(OUTDIR/"gating_per_sample.csv")    # fold,image,true,pred,p1,correct,...
print("test", len(test), "| TAU", TAU, "| cols:", list(test.columns))


In [ ]:
# Cell 9. 固定阈值 -> 逐折(自动正确/自动错误/转人工) + 整体表现表
N=len(test); A=test[test.p1>=TAU]
print(f"[pooled test @ TAU={TAU}]")
print(f"  auto_cov={len(A)/N*100:.1f}% ({len(A)}/{N}) | auto_acc={A['correct'].mean()*100:.1f}% | "
      f"manual={N-len(A)} | leaked_err={int((A['correct']==0).sum())} | baseline_no_gate={test['correct'].mean()*100:.1f}%\n")
rows=[]
for k in sorted(test['fold'].unique()):
    te=test[test.fold==k]; a=te[te.p1>=TAU]
    ac=int((a['correct']==1).sum()); aw=int((a['correct']==0).sum())
    rows.append({'fold':k,'n':len(te),'auto':len(a),'auto_correct':ac,'auto_wrong':aw,
                 'manual':len(te)-len(a),'auto_cov%':round(len(a)/len(te)*100,1),
                 'auto_acc%':round(ac/len(a)*100,1) if len(a) else float('nan')})
r=pd.DataFrame(rows); print(r.to_string(index=False))
print(f"\nper-fold: cov {r['auto_cov%'].mean():.1f}%+-{r['auto_cov%'].std():.1f}% | "
      f"acc {r['auto_acc%'].mean():.1f}%+-{r['auto_acc%'].std():.1f}% | auto_wrong total {r['auto_wrong'].sum()}")
r.to_csv(OUTDIR/"per_fold_summary.csv", index=False); print(">>> saved per_fold_summary.csv")


In [ ]:
# Cell 10. 导出: 全样本分档表 + 给 Stage 2 同学的 (进入自动档却判错) 清单
t = test.copy()
t["tier"]=np.where(t.p1>=TAU,"auto","manual")
t["true_name"]=t["true"].map(lambda i:CLASS_NAMES[i])
t["pred_name"]=t["pred"].map(lambda i:CLASS_NAMES[i])
base=[c for c in ["fold","image","p1","tier","correct","true_name","pred_name"] if c in t.columns]
t[base].sort_values(["fold","tier"]).to_csv(OUTDIR/"per_sample_tiers.csv", index=False)
print(">>> saved per_sample_tiers.csv (all samples, tier/correct/true->pred)")

err = t[(t.tier=="auto") & (t.correct==0)].copy()
err["true_repair"]=err["true_name"].map(M2REP)
err["pred_repair"]=err["pred_name"].map(M2REP)
err["repair_changed"]=(err["true_repair"]!=err["pred_repair"])
ec=[c for c in ["fold","image","p1","true_name","pred_name","true_repair","pred_repair","repair_changed"] if c in err.columns]
out=err[ec].sort_values(["fold","true_name"]).reset_index(drop=True)
print(f"\nentering Stage 2 but wrong = {len(out)} (per fold); repair changed {int(out['repair_changed'].sum())} ({out['repair_changed'].mean()*100:.0f}%)")
print(out.to_string(index=False))
out.to_csv(OUTDIR/"stage2_auto_errors.csv", index=False)
print(">>> saved stage2_auto_errors.csv (send to Stage 2 colleague)")
